In [1]:
import numpy as np
import pandas as pd
import re

In [24]:
def parse_age_to_interval(age_str):
    # "0 to 4 years" → Interval(0, 4, closed='left')
    m = re.match(r'(\d+) to (\d+)', age_str)
    if m:
        return pd.Interval(int(m.group(1)), int(m.group(2)) + 1, closed='left')
    
    # "100+" → Interval(100, inf, closed='left')
    m = re.match(r'(\d+) years and ', age_str)
    if m:
        return pd.Interval(int(m.group(1)), np.inf, closed='left')
    
    return pd.NA

In [46]:
age_sex = pd.read_csv(
    '../data/bayesian_network/province-age-sex.csv',
    skiprows=[0,1,2,3,4,5,6,7,10,11],
    header=[0,1],
    index_col=0,
)[0:21]
age_sex.columns.names = ['geography', 'gender']
age_sex.index.name = 'age_group'

geo = (
    age_sex.columns.get_level_values('geography')
    .to_series()
    .replace(r'^Unnamed.*', pd.NA, regex=True)
    .ffill()
)
age_sex.columns = pd.MultiIndex.from_arrays(
    [geo.values, age_sex.columns.get_level_values('gender')],
    names=['geography', 'gender']
)

age_sex = age_sex.apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', ''), errors='coerce'))
age_sex.index = age_sex.index.map(parse_age_to_interval)

In [47]:
result = (
    age_sex
    .stack(['geography', 'gender'])
    .unstack('geography')
    .swaplevel('age_group', 'gender')
    .sort_index()
)
result['Canada (excluding territories)'] = result.sum(axis=1)
result = result.rename(
    columns = {
        'Canada (except provinces)': 'Canada (excluding territories)'
    },
    index = {
        'Men+': 'Male',
        'Women+': 'Female'
    }
)

In [48]:
result

geography             Alberta  British Columbia  Manitoba  New Brunswick  \
gender age_group                                                           
Male   [0.0, 5.0)      131507            114559     42204          17188   
       [5.0, 10.0)     146831            130524     46198          20019   
       [10.0, 15.0)    147785            136018     45458          21147   
       [15.0, 20.0)    137819            143876     44308          21076   
       [20.0, 25.0)    142181            172054     56853          23635   
       [25.0, 30.0)    156448            205321     54726          24116   
       [30.0, 35.0)    175956            210865     51332          23779   
       [35.0, 40.0)    183812            197455     49338          24432   
       [40.0, 45.0)    169950            176719     45948          24694   
       [45.0, 50.0)    149538            159522     42121          25179   
       [50.0, 55.0)    139375            165387     40380          26545   
       [55.0, 60.0)    137765            173844     42395          30300   
       [60.0, 65.0)    137959            176121     43588          31198   
       [65.0, 70.0)    115285            158345     37206          28824   
       [70.0, 75.0)     82947            132486     28736          24013   
       [75.0, 80.0)     55748             94164     20286          16670   
       [80.0, 85.0)     33210             57626     12130           9774   
       [85.0, 90.0)     18600             32340      6451           4803   
       [90.0, 95.0)      7598             13816      2689           1809   
       [95.0, 100.0)     1572              2990       688            354   
       [100.0, inf)       151               312        74             33   
Female [0.0, 5.0)      125380            109755     40567          16267   
       [5.0, 10.0)     139035            123942     43211          19412   
       [10.0, 15.0)    138576            127288     42745          20445   
       [15.0, 20.0)    129966            136564     41119          19981   
       [20.0, 25.0)    133646            167361     48779          21971   
       [25.0, 30.0)    149543            194649     48077          22426   
       [30.0, 35.0)    173723            203362     48182          23626   
       [35.0, 40.0)    179643            192885     47045          24940   
       [40.0, 45.0)    164595            177099     44613          24962   
       [45.0, 50.0)    144058            166307     41102          26199   
       [50.0, 55.0)    135343            175691     40225          26776   
       [55.0, 60.0)    136210            180918     42857          30974   
       [60.0, 65.0)    137882            186585     43994          32475   
       [65.0, 70.0)    117424            171421     38557          30510   
       [70.0, 75.0)     87964            143067     31755          25649   
       [75.0, 80.0)     61636            103752     23008          18409   
       [80.0, 85.0)     40305             68388     15603          11899   
       [85.0, 90.0)     26323             43004     10318           7272   
       [90.0, 95.0)     14223             23484      6133           3649   
       [95.0, 100.0)     4417              7624      2083           1220   
       [100.0, inf)       802              1355       416            219   

geography             Newfoundland and Labrador  Nova Scotia  Ontario  \
gender age_group                                                        
Male   [0.0, 5.0)                         10041        21563   365599   
       [5.0, 10.0)                        11924        24838   402965   
       [10.0, 15.0)                       13197        26219   419881   
       [15.0, 20.0)                       14026        26067   443227   
       [20.0, 25.0)                       15191        33807   539339   
       [25.0, 30.0)                       14370        35632   601163   
       [30.0, 35.0)                       14988        33721   570277  

In [49]:
result.to_csv('../data/bayesian_network/age-fixed.csv')

------------------------------------

In [50]:
data2 = pd.read_csv(
    '../data/bayesian_network/age-sex-health.csv',
    skiprows=[0,1,2,3,4,5,6,7,8,11,12],
    header=[0,1],
    index_col=0
)[:4]
data2.columns.names = ['age_group', 'sex']
data2.index.name = 'condition'

filled_columns = (
    data2.columns.get_level_values('age_group')
    .to_series()
    .replace(r'^Unnamed.*', pd.NA, regex=True)
    .ffill()
)

data2.columns = pd.MultiIndex.from_arrays(
    [filled_columns.values, data2.columns.get_level_values('sex')],
    names=['age_group', 'sex']
)

data2 = data2.apply(lambda col: col.astype(str).str.replace('E', ''))
data2 = data2.apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', ''), errors='coerce'))
data2 = data2.drop(columns=['Total, 18 years and over'])

C:\Users\ltyih\AppData\Local\Temp\ipykernel_24812\1872592946.py:24: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  data2 = data2.drop(columns=['Total, 18 years and over'])


In [51]:
data2.columns = pd.MultiIndex.from_arrays(
    [data2.columns.get_level_values('age_group').map(parse_age_to_interval),
     data2.columns.get_level_values('sex')],
    names=['age_group', 'sex']
)

In [52]:
data2

age_group                                          [18.0, 35.0)           \
sex                                                       Males  Females   
condition                                                                  
Body mass index, adjusted self-reported, adult ...       955200   974800   
High blood pressure 21 22                                122900    84600   
Current smoker, daily or occasional 23 24 25 26 27       562800   330200   
Influenza immunization in the past 12 months 28 29       647100  1037900   

age_group                                          [35.0, 50.0)           \
sex                                                       Males  Females   
condition                                                                  
Body mass index, adjusted self-reported, adult ...      1258100  1132100   
High blood pressure 21 22                                442000   308100   
Current smoker, daily or occasional 23 24 25 26 27       626900   425100   
Influenza immunization in the past 12 months 28 29       812500  1181800   

age_group                                          [50.0, 65.0)           \
sex                                                       Males  Females   
condition                                                                  
Body mass index, adjusted self-reported, adult ...      1284600  1185800   
High blood pressure 21 22                               1130600   913300   
Current smoker, daily or occasional 23 24 25 26 27       649900   572000   
Influenza immunization in the past 12 months 28 29      1215600  1454300   

age_group                                          [65.0, inf)           
sex                                                      Males  Females  
condition                                                                
Body mass index, adjusted self-reported, adult ...      869500  1045400  
High blood pressure 21 22                              1440800  1700900  
Current smoker, daily or occasional 23 24 25 26 27      325600   304300  
Influenza immunization in the past 12 months 28 29     1932500  2284600

In [53]:
pivot2 = (
    data2
    .stack(['age_group', 'sex'])
    .unstack(['condition'])
    .swaplevel('age_group', 'sex')
    .sort_index()
)

pivot2 = pivot2.rename(
    columns = {
        'Body mass index, adjusted self-reported, adult (18 years and over), obese 15 16 17 18 19 20': 'Obese',
        'High blood pressure 21 22': 'High blood pressure',
        'Current smoker, daily or occasional 23 24 25 26 27': 'Current smoker',
        'Influenza immunization in the past 12 months 28 29': 'Recently vaccinated'
    },
    index = {
        'Males': 'Male',
        'Females': 'Female'
    }
)
pivot2

condition              Obese  High blood pressure  Current smoker  \
sex    age_group                                                    
Female [18.0, 35.0)   974800                84600          330200   
       [35.0, 50.0)  1132100               308100          425100   
       [50.0, 65.0)  1185800               913300          572000   
       [65.0, inf)   1045400              1700900          304300   
Male   [18.0, 35.0)   955200               122900          562800   
       [35.0, 50.0)  1258100               442000          626900   
       [50.0, 65.0)  1284600              1130600          649900   
       [65.0, inf)    869500              1440800          325600   

condition            Recently vaccinated  
sex    age_group                          
Female [18.0, 35.0)              1037900  
       [35.0, 50.0)              1181800  
       [50.0, 65.0)              1454300  
       [65.0, inf)               2284600  
Male   [18.0, 35.0)               647100  
       [35.0, 50.0)               812500  
       [50.0, 65.0)              1215600  
       [65.0, inf)               1932500

In [54]:
pivot2.to_csv('../data/bayesian_network/health-fixed.csv', index=True)